# Обучение 

In [14]:
import pandas as pd
import sys
from pathlib import Path
from matplotlib import pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import PolynomialFeatures

sys.path.append('..')
from src.data_loader import load_data
from src.preprocessing import build_preprocessor, get_train_test
from src.train import train, model_dump
from src.predict import get_predict, get_metrics

In [2]:
file_path = Path('../data/mental_health.csv').resolve()
df = load_data(file_path)
df.head()

,Person_ID,Age,Gender,Occupation,Daily_Screen_Time,Social_Media_Usage,Night_Usage,Sleep_Hours,Stress_Level,Work_Study_Hours,Physical_Activity,Social_Interaction_Score,Caffeine_Intake,Smoking,Alcohol,Depression,Anxiety,Burnout
0,1,54,Male,Student,10.2,7.5,0,6.5,8,5.8,Low,6,2,0,1,1,1,1
1,2,44,Male,Student,6.8,4.5,0,5.1,4,7.9,High,2,1,1,0,0,0,0
2,3,30,Male,Employed,5.5,6.9,0,3.5,10,9.4,Low,2,0,1,1,1,0,1
3,4,58,Male,Employed,5.6,4.1,0,9.0,2,2.0,High,10,0,0,1,0,0,0
4,5,23,Female,Employed,10.1,6.0,1,3.8,4,4.6,Low,4,4,0,1,1,1,0


In [3]:
numeric_df = df.select_dtypes(include="number").copy()
numeric_df.drop(["Person_ID", "Depression", "Anxiety", "Burnout"] , inplace=True, axis=1)
numeric_features = numeric_df.columns
numeric_features

Index(['Age', 'Daily_Screen_Time', 'Social_Media_Usage', 'Night_Usage',
       'Sleep_Hours', 'Stress_Level', 'Work_Study_Hours',
       'Social_Interaction_Score', 'Caffeine_Intake', 'Smoking', 'Alcohol'],
      dtype='str')

In [4]:
categorical_df = df.select_dtypes(include="string").copy()

categorical_features = categorical_df.columns
categorical_features

Index(['Gender', 'Occupation', 'Physical_Activity'], dtype='str')

In [5]:


preprocessor = build_preprocessor(numeric_features,
                                  categorical_features)

pipe_log_reg = Pipeline(
    [
        ("preprocessor", preprocessor),
        ("model", LogisticRegression())

    ]
)

In [6]:
param_grid = [
      {
          "model__C": [0.01, 0.1, 1, 10],
          "model__penalty": ["l1"],
          "model__solver": ["liblinear"],
          "preprocessor__num__poly__degree": [1, 2, 3]
      },
      {
          "model__C": [0.01, 0.1, 1, 10],
          "preprocessor__num__poly__degree": [1, 2, 3],
          "model__penalty": ["l2"],
          "model__solver": ["liblinear", "lbfgs"]
      }
  ]


In [7]:
df.head()

,Person_ID,Age,Gender,Occupation,Daily_Screen_Time,Social_Media_Usage,Night_Usage,Sleep_Hours,Stress_Level,Work_Study_Hours,Physical_Activity,Social_Interaction_Score,Caffeine_Intake,Smoking,Alcohol,Depression,Anxiety,Burnout
0,1,54,Male,Student,10.2,7.5,0,6.5,8,5.8,Low,6,2,0,1,1,1,1
1,2,44,Male,Student,6.8,4.5,0,5.1,4,7.9,High,2,1,1,0,0,0,0
2,3,30,Male,Employed,5.5,6.9,0,3.5,10,9.4,Low,2,0,1,1,1,0,1
3,4,58,Male,Employed,5.6,4.1,0,9.0,2,2.0,High,10,0,0,1,0,0,0
4,5,23,Female,Employed,10.1,6.0,1,3.8,4,4.6,Low,4,4,0,1,1,1,0


In [8]:
Xtrain, Xtest, ytrain, ytest = get_train_test(df, target_col="Depression", to_drop=["Anxiety", "Burnout"])

In [9]:
gs = train(Xtrain, ytrain, pipe_log_reg, params_grid=param_grid)
gs.best_params_


c:\Users\Bogdan\AppData\Local\pypoetry\Cache\virtualenvs\mental-health-3dElcIj1-py3.12\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


{'model__C': 0.01,
 'model__penalty': 'l2',
 'model__solver': 'lbfgs',
 'preprocessor__num__poly__degree': 1}

In [10]:
file_path = Path('../models/logistic_regression.pkl').resolve()
model_dump(gs.best_estimator_, file_path)

In [18]:
y_predict = get_predict(gs.best_estimator_, Xtest, return_proba=False)
y_proba = get_predict(gs.best_estimator_, Xtest, return_proba=True)

metrics = get_metrics(ytest, y_predict, y_proba)

metrics

{'accuracy': 0.6,
 'precision': 0.5980392156862745,
 'recall': 0.61,
 'f1': 0.6039603960396039,
 'roc_auc': 0.63055}